# B6: Prediction Reasoning — Requirement Extraction Interpretability

**Deliverable B Notebooks — Part B6**

This notebook demonstrates **human-understandable reasoning** for individual requirement predictions produced by the PURE extraction system. For each extracted requirement, we trace:

- **Source Traceability**: The exact SRS verse that triggered extraction
- **Classification Rationale**: Why the requirement was classified as Functional (FR) vs Non-Functional (NFR)
- **Confidence Assessment**: Which fields are uncertain and why

---

### Why Reasoning Matters

When an LLM extracts requirements from a raw SRS document, stakeholders — especially non-technical business analysts — need to understand **why** the system made each decision. Without explanation, extracted requirements are black boxes: you see the output but cannot verify its fidelity to the source text.


## 1. Setup and Data Loading

Load the PURE output datasets containing LLM-extracted requirements.

In [ ]:
import json
import os
from IPython.display import HTML, display
import pandas as pd

# Base directory for PURE output datasets
PURE_BASE = "/Users/xd/Final_Project/Final Project/raw/datasets/PURE_output"

# Load Get Real dataset
get_real_path = os.path.join(PURE_BASE, "2007 - get real 0.2", "requirements.json")
with open(get_real_path, "r", encoding="utf-8") as f:
    get_real = json.load(f)
print(f"Get Real 0.2 dataset: {len(get_real)} requirements")

# Load Mashboot dataset
mashboot_path = os.path.join(PURE_BASE, "2010 - mashboot", "requirements.json")
with open(mashboot_path, "r", encoding="utf-8") as f:
    mashboot = json.load(f)
print(f"Mashboot dataset: {len(mashboot)} requirements")


## 2. The Reasoning Framework

We evaluate each extracted requirement along **three dimensions** of interpretability:

### Dimension 1: Source Traceability
Maps every extracted requirement back to the **exact original SRS verse(s)**. A strong trace means:
- The source verse is explicitly quoted in `source_location.verse`
- The requirement description semantically aligns with the source text
- No claims are made that contradict or substantially exceed the source

**Traceability scoring:**
> **High** — Source verse directly maps, description is faithful paraphrase
> **Medium** — Source verse present, some elaboration or inference added
> **Low** — Source verse vague or extracted description goes beyond source

---

### Dimension 2: Classification Rationale (FR vs NFR)
Explains **why** the LLM assigned `functional` vs `non-functional`:
- **Functional**: Describes what the system **shall do** — a behavior, action, or capability the user can invoke or observe
- **Non-Functional**: Describes **how well** the system shall do something — quality attributes (performance, security, usability, compatibility)

**Classification signals the LLM uses:**
> ✅ Strong FR signals: "shall provide", "shall display", "shall allow user to..."
> ✅ Strong NFR signals: "shall respond within X seconds", "shall be compatible with...", "shall use encryption"
> ⚠️ Ambiguous: Requirements that blend behavior with quality constraints

---

### Dimension 3: Confidence Assessment
Evaluates confidence per key field:

| Field | High Confidence Indicator | Low Confidence Indicator |
|---|---|---|
| **Title** | Concise, derived from clear noun phrase in source | Invented or too generic (e.g., 'System feature') |
| **Description** | Faithful paraphrase with domain terminology | Heavily elaborated beyond source |
| **Type** | Clear FR/NFR signals in source | Mixed signals, could be either FR or NFR |
| **Acceptance Criteria** | Testable, specific, tied to source claims | Vague, unverifiable, or not grounded in source |
| **Source Location** | Exact verse quoted | No verse found, or verse does not match description |

**Confidence Flags:**
> 🟢 **Clear**: Single interpretation, strong source alignment
> 🟡 **Caution**: Some inference or interpretation needed
> 🟠 **Ambiguous**: Multiple valid readings possible

## 3. Example 1 — Functional Requirement (Get Real)

We examine **REQ-002** from the Get Real 0.2 dataset: a functional requirement about the High School Courses page.

In [ ]:
# Load Example 1: Functional requirement from Get Real
ex1 = get_real[1]  # REQ-002
print("=" * 80)
print("EXAMPLE 1: Functional Requirement")
print("=" * 80)
print(json.dumps(ex1, indent=2))


### 3.1 Full Requirement JSON

The extracted requirement contains:
- **ID**: REQ-002
- **Title**: High School Courses page
- **Description**: The site shall provide a High School Courses page showing recommended curricula...
- **Type**: functional
- **Acceptance Criteria**: 5 testable criteria
- **Source Location**: Original SRS verse

In [ ]:
# Reasoning Analysis for Example 1
print("=" * 80)
print("REASONING ANALYSIS — Example 1 (Functional: REQ-002)")
print("=" * 80)

# Dimension 1: Source Traceability
print("")
print("1. SOURCE TRACEABILITY")
print("-" * 40)
print("Source verse:")
print(f"  {ex1['source_location']['verse'][0]}")
print("Traceability: HIGH")
print("Reason: The source verse explicitly names three entities (OUS campuses, other")
print("university sources, ACM) that all appear in both the description and")
print("acceptance criteria. Every claim in the extracted requirement maps")
print("directly to a clause in the source verse.")

# Dimension 2: Classification Rationale
print("")
print("2. CLASSIFICATION RATIONALE (FR vs NFR)")
print("-" * 40)
print("Assigned type: functional")
print("Confidence: HIGH")
print("Reasoning:")
print("- The source describes a CONCRETE SYSTEM BEHAVIOR: providing a page")
print("  that displays specific content to users.")
print("- Key verb: 'show' — this is what the system SHALL DO (action)")
print("- This is NOT about how well the page performs, but that it EXISTS")
print("  and provides specific information.")
print("- The phrase 'The intent of this section is to show...' is a clear")
print("  functional directive despite using 'intent' wording.")
print("- Verdict: Unambiguously functional — describes system capability.")

# Dimension 3: Confidence Assessment
print("")
print("3. CONFIDENCE ASSESSMENT")
print("-" * 40)
fields1 = {
    "Title": ("High", "Title derived directly from described section in source."),
    "Description": ("High", "Faithful paraphrase. Source says 'show recommended curricula'; description preserves this with 'showing recommended curricula'. Adds reasonable structure."),
    "Type": ("High", "Clear functional signals: system provides a page with content. No quality attributes involved."),
    "Acceptance Criteria": ("Medium", "Most criteria directly derived from source clauses. The criterion about 'UO is currently developing HS course recommendations' is slight reframing of 'UO is working to develop...'"),
    "Source Location": ("High", "Exact SRS verse preserved verbatim.")
}
for field, (conf, reason) in fields1.items():
    print(f"  {field}: {conf} — {reason}")

print("")
print("4. NON-TECHNICAL EXPLANATION")
print("-" * 40)
print("This requirement was extracted because the original SRS document explicitly")
print('stated that a High School Courses section should "show recommended curricula"')
print("for students interested in computer science.")
print("")
print("The LLM correctly identified this as a FUNCTIONAL requirement because:")
print("- It describes something the website must DO (show curriculum information)")
print("- A user visiting the site would interact with this page")
print("- It is NOT about how fast or secure the page is")
print("")
print("Confidence is HIGH overall because the source text was specific and direct.")
print("The only small caveat is that one acceptance criterion slightly reframes")
print("the source language rather than using it verbatim.")


## 4. Example 2 — Non-Functional Requirement (Get Real)

We examine **REQ-004** from the Get Real 0.2 dataset: a non-functional requirement about server performance.

In [ ]:
# Load Example 2: Non-functional requirement from Get Real
ex2 = get_real[3]  # REQ-004
print("=" * 80)
print("EXAMPLE 2: Non-Functional Requirement")
print("=" * 80)
print(json.dumps(ex2, indent=2))


### 4.1 Full Requirement JSON

Extracted requirement REQ-004 specifies server performance as a quality constraint on the hosting infrastructure.

In [ ]:
# Reasoning Analysis for Example 2
print("=" * 80)
print("REASONING ANALYSIS — Example 2 (Non-Functional: REQ-004)")
print("=" * 80)

# Dimension 1: Source Traceability
print("")
print("1. SOURCE TRACEABILITY")
print("-" * 40)
print("Source verse:")
print(f"  {ex2['source_location']['verse'][0]}")
print("Traceability: HIGH")
print("Reason: The source verse is reproduced nearly word-for-word in the")
print("description. The original SRS contains three distinct claims")
print("(adequate response time, student attention span, OUS benchmark)")
print("all of which are preserved in the extracted requirement.")

# Dimension 2: Classification Rationale
print("")
print("2. CLASSIFICATION RATIONALE (FR vs NFR)")
print("-" * 40)
print("Assigned type: non-functional")
print("Confidence: HIGH")
print("Reasoning:")
print("- This is a classic NFR: it specifies HOW WELL the system must")
print("  perform (server response time), not WHAT behavior it exhibits.")
print("- The requirement does NOT add a new feature — it constrains the")
print("  existing hosting environment.")
print("- Key phrase: 'adequate response time' — a quality attribute.")
print("- The user-facing justification ('short attention spans') provides")
print("  business rationale but does not change the requirement type.")
print("- Comparison to OUS site as a BENCHMARK is a common NFR pattern:")
print("  defining quality by reference to a known standard.")
print("- Verdict: Unambiguously non-functional — quality/constraint.")

# Dimension 3: Confidence Assessment
print("")
print("3. CONFIDENCE ASSESSMENT")
print("-" * 40)
fields2 = {
    "Title": ("High", "'Server performance' is precise and standard for this domain."),
    "Description": ("High", "Faithful to source. Preserves all three claims from original verse."),
    "Type": ("High", "Performance constraint is textbook NFR — unambiguous."),
    "Acceptance Criteria": ("Medium", "'Adequate response time' is inherently vague — no specific numeric threshold given. The benchmark comparison to OUS site helps but remains subjective. A BA should note this needs quantification later."),
    "Source Location": ("High", "Exact verse captured verbatim.")
}
for field, (conf, reason) in fields2.items():
    print(f"  {field}: {conf} — {reason}")

print("")
print("4. NON-TECHNICAL EXPLANATION")
print("-" * 40)
print("This requirement was extracted because the original SRS explicitly stated")
print('that the server must provide "adequate response time" — the target users')
print("(high school students) have short attention spans and will leave if pages")
print("load slowly.")
print("")
print("The LLM correctly classified this as NON-FUNCTIONAL because:")
print("- It does not describe a new feature or user action")
print("- It specifies a QUALITY ATTRIBUTE (how fast the server responds)")
print("- The system must 'be hosted on a server that provides...' — this is a")
print("  constraint on the infrastructure, not a behavior")
print("")
print("Confidence is HIGH for everything EXCEPT the acceptance criteria.")
print("WARNING: The acceptance criteria say 'adequate response time' without")
print("specifying a number. A business analyst should flag: 'adequate' is too")
print("vague to test. Consider replacing with 'response time under 2 seconds'.")


## 5. Example 3 — Mashboot Requirement

We examine **REQ-005** from the Mashboot dataset: a non-functional requirement about backup operations.

In [ ]:
# Load Example 3: Requirement from Mashboot
ex3 = mashboot[4]  # REQ-005
print("=" * 80)
print("EXAMPLE 3: Mashboot Requirement (Non-Functional: Backup)")
print("=" * 80)
print(json.dumps(ex3, indent=2))


### 5.1 Full Requirement JSON

Extracted requirement REQ-005 specifies backup operation constraints with measurable availability guarantees.

In [ ]:
# Reasoning Analysis for Example 3
print("=" * 80)
print("REASONING ANALYSIS — Example 3 (Mashboot REQ-005: Backup)")
print("=" * 80)

# Dimension 1: Source Traceability
print("")
print("1. SOURCE TRACEABILITY")
print("-" * 40)
print("Source verse:")
print(f"  {ex3['source_location']['verse'][0]}")
print("")
print("Traceability: HIGH")
print("Reason: The extracted requirement covers claims from the source verse:")
print("  [OK] 'backup of the data system to local or remote non-volatile storage'")
print("        -> maps to acceptance criterion about backup storage")
print("  [OK] 'Incremental backups should not create outages'")
print("        -> maps to 'Incremental backups cause no outages'")
print("  [OK] 'full backups do not interfere with user interaction for more than")
print("        10 minutes'")
print("        -> maps directly to an acceptance criterion")
print("  [OK] 'Priority 1' indicates highest priority requirement")
print("")
print("Note: The source verse contains structural artifacts ('0630', 'Priority 1')")
print("from the SRS numbering system. The LLM cleaned these appropriately.")

# Dimension 2: Classification Rationale
print("")
print("2. CLASSIFICATION RATIONALE (FR vs NFR)")
print("-" * 40)
print("Assigned type: non-functional")
print("Confidence: MEDIUM-HIGH")
print("Reasoning:")
print("- This is an AVAILABILITY and RELIABILITY constraint — a classic NFR.")
print("- The requirement constrains HOW the backup system behaves during")
print("  operation (must not cause outages, must minimize interference).")
print("- It does NOT describe a new user-facing feature.")
print("- The SPECIFIC NUMERIC THRESHOLD ('10 minutes') is a strong NFR signal.")
print("- Potential ambiguity: 'Incremental backups should not create outages'")
print("  uses 'should' instead of 'shall' — weaker commit language. This")
print("  introduces slight uncertainty about whether this is truly mandatory.")
print("- Verdict: Non-functional (reliability/availability constraint).")

# Dimension 3: Confidence Assessment
print("")
print("3. CONFIDENCE ASSESSMENT")
print("-" * 40)
fields3 = {
    "Title": ("High", "'Backup non-interference' is clear and descriptive."),
    "Description": ("High", "Combines the two backup-related constraints from source into a coherent single requirement. Faithful to source."),
    "Type": ("Medium", "Availability/reliability is clearly NFR. However, the source uses 'should' (non-binding) rather than 'shall' (binding), which introduces ambiguity about whether this is truly mandatory."),
    "Acceptance Criteria": ("High", "Exceptionally well-defined. The '10 minutes' threshold is specific and testable. 'No outages' for incremental backups is also verifiable. This is one of the best-specified NFRs in the dataset."),
    "Source Location": ("High", "Full verse captured including numbering artifacts.")
}
for field, (conf, reason) in fields3.items():
    print(f"  {field}: {conf} — {reason}")

print("")
print("4. NON-TECHNICAL EXPLANATION")
print("-" * 40)
print("This requirement was extracted because the original SRS specified that")
print("backup operations must not disrupt the system:")
print("- Incremental backups: ZERO downtime allowed")
print("- Full backups: maximum 10 minutes of user disruption")
print("")
print("The LLM classified this as NON-FUNCTIONAL because:")
print("- It constrains system RELIABILITY, not a new feature")
print("- It defines availability thresholds (how much downtime is acceptable)")
print("- Users do not 'use' backups — they experience them when things fail")
print("")
print("Confidence is HIGH for most fields. The one caution:")
print("- The source uses 'should not create outages' instead of 'shall not'")
print("- This means the vendor might argue it is a recommendation, not a mandate")
print("- A business analyst should clarify this with stakeholders")
print("")
print("GOOD NEWS: The 10-minute threshold is specific and testable — this is")
print("an excellent example of a well-defined NFR with clear acceptance criteria.")


## 6. Summary Table — All Three Examples

Consolidated view of the reasoning analysis across all three examples.

In [ ]:
# Build summary table
data = [
    {
        "ID": ex1["requirement_id"],
        "Title": ex1["title"],
        "Source Verse": ex1["source_location"]["verse"][0][:100] + "...",
        "Type": ex1["type"].upper(),
        "Type Rationale": "Describes system BEHAVIOR: providing a page with curriculum content. Clear 'what the system does' language. User-facing feature.",
        "Confidence Level": "HIGH",
        "Confidence Flags": "AC slightly reframes source (UO developing criterion)"
    },
    {
        "ID": ex2["requirement_id"],
        "Title": ex2["title"],
        "Source Verse": ex2["source_location"]["verse"][0][:100] + "...",
        "Type": ex2["type"].upper(),
        "Type Rationale": "Specifies a QUALITY ATTRIBUTE: server response time. Constraint on infrastructure, not a behavior. Performance requirement.",
        "Confidence Level": "HIGH",
        "Confidence Flags": "AC lacks numeric threshold ('adequate' is vague)"
    },
    {
        "ID": ex3["requirement_id"],
        "Title": ex3["title"],
        "Source Verse": ex3["source_location"]["verse"][0][:100] + "...",
        "Type": ex3["type"].upper(),
        "Type Rationale": "AVAILABILITY/RELIABILITY constraint: backup must not cause outages. Numeric threshold (10 min) confirms NFR.",
        "Confidence Level": "MEDIUM-HIGH",
        "Confidence Flags": "Source uses 'should' not 'shall' — binding strength uncertain"
    }
]

df = pd.DataFrame(data)
display(HTML("<h3>Reasoning Summary — Three Extraction Examples</h3>"))
display(HTML(df.to_html(index=False, escape=False)))


### 6.1 Confidence Heatmap by Field

Visual summary of confidence levels across all fields for the three examples.

In [ ]:
# Build confidence heatmap
confidence_data = [
    {
        "Requirement": "REQ-002 (Get Real)",
        "Title": "HIGH",
        "Description": "HIGH",
        "Type": "HIGH",
        "Acceptance Criteria": "MEDIUM",
        "Source Location": "HIGH"
    },
    {
        "Requirement": "REQ-004 (Get Real)",
        "Title": "HIGH",
        "Description": "HIGH",
        "Type": "HIGH",
        "Acceptance Criteria": "MEDIUM",
        "Source Location": "HIGH"
    },
    {
        "Requirement": "REQ-005 (Mashboot)",
        "Title": "HIGH",
        "Description": "HIGH",
        "Type": "MEDIUM-HIGH",
        "Acceptance Criteria": "HIGH",
        "Source Location": "HIGH"
    }
]

conf_df = pd.DataFrame(confidence_data)
display(HTML("<h3>Confidence Heatmap by Field</h3>"))
display(HTML(conf_df.to_html(index=False)))


## 7. Discussion — How Reasoning Helps Business Analysts

### 7.1 Building Trust Through Transparency

When a business analyst (BA) reviews LLM-extracted requirements, they need to answer: **"Can I trust this?"** Without reasoning, every extracted requirement requires re-reading the original SRS to verify. This defeats the purpose of automated extraction.

The three-dimensional reasoning framework provides:

| Dimension | What the BA Learns | Action If Concern |
|---|---|---|
| **Source Traceability** | "Did the LLM invent this, or is it really in the SRS?" | If traceability is low, verify against source document directly |
| **Classification Rationale** | "Is this categorized correctly as FR or NFR?" | Misclassification affects sprint planning, testing strategy, and budget |
| **Confidence Assessment** | "Which fields need my attention?" | Focus review effort on yellow/red fields, skip green fields |

### 7.2 Practical Impact on Review Efficiency

**Without reasoning:** A BA must re-read the entire SRS for every extracted requirement (high effort).

**With reasoning:** The BA reviews only the flagged fields:
> - REQ-002: Check the UO development criterion against source
> - REQ-004: Note that 'adequate response time' needs quantification
> - REQ-005: Clarify whether 'should' means mandatory or recommended

This reduces review time from **hours per requirement** to **minutes per flagged field**.

### 7.3 Enabling Accountability

When reasoning is recorded:

1. **Audit trail**: Every requirement has a documented link to its source verse, enabling compliance verification in regulated industries.

2. **Dispute resolution**: When stakeholders disagree about a requirement, the reasoning shows: "This is what the original SRS said, and this is why the system classified it this way."

3. **Iterative improvement**: Low-confidence flags identify cases where the LLM prompt or training data should be improved.

4. **Onboarding**: New team members can understand requirements without reading the full SRS — the reasoning provides context.

### 7.4 Key Insights from Our Examples

**REQ-002 (Functional — High School Courses)**: The easiest case to trust. Source verse is explicit, classification is unambiguous (it clearly describes a page the system provides), and the description is a faithful paraphrase. A BA reviewing this should feel very confident.

**REQ-004 (Non-Functional — Server Performance)**: The acceptance criteria confidence flag is the most important output. The requirement itself is well-extracted, but a BA must know that 'adequate response time' is not testable in its current form. The system identifies this concern rather than hiding it.

**REQ-005 (Non-Functional — Backup)**: This example demonstrates that even well-extracted requirements can have binding-strength ambiguity. The 'should' vs 'shall' distinction is something a non-technical reviewer would likely miss without the reasoning framework flagging it.

### 7.5 Recommendations for BA Workflow

1. **Sort by confidence flags**: Review requirements with medium or low confidence flags first.

2. **Focus on acceptance criteria**: This field most often needs human judgment — the LLM may fabricate or over-specify.

3. **Use type rationale for classification audits**: When requirements are mis-typed (e.g., security requirements labeled functional instead of non-functional), the rationale explains the LLM's logic, enabling targeted corrections.

4. **Treat 'should' in source as a red flag**: When the original SRS uses 'should', 'may', or 'might' instead of 'shall', the extracted requirement inherits this ambiguity. These should always be flagged for stakeholder clarification.

---

*This notebook demonstrates that interpretable AI for requirement extraction is not an academic luxury — it is a practical necessity for business stakeholders who need to verify, trust, and act on automated extraction results.*